# 2.7 Mutability, Copying, Nesting & Unpacking

**Prerequisites:** 2.1–2.6 (all datatype notebooks)  
**Target:** Python 3.12+ (notes flag 3.13/3.14 differences)

### What you'll learn
- Names, objects and values — what assignment actually does
- Aliasing: why changing one variable changes another
- Mutable vs immutable, and why it decides almost everything else
- Shallow copy vs `copy.deepcopy()`
- The mutable default argument trap
- Hashability — the rule behind dict keys and set elements
- Building and safely navigating nested structures
- Unpacking in every form Python offers

---

## Why this notebook exists

You have now met every built-in container: `str`, `tuple`, `list`, `dict`, `set`. Each
notebook mentioned mutability in passing. This one takes the cross-cutting ideas that
connect them and treats them properly, because they are the source of a large share of
real Python bugs:

- Why did changing `b` also change `a`?
- Why does `.copy()` sometimes not copy?
- Why can a tuple be a dict key but not a list?
- Why did my function "remember" the last call's data?

All four have the same root cause, and it is worth understanding once, properly.

---

## 1. Names, objects and values

In many languages, a variable is a **box** holding a value. Assignment copies a value into
the box.

**Python does not work this way.** In Python:

- An **object** lives in memory. It has a type, a value, and an identity.
- A **name** is a *label* attached to an object.
- Assignment (`x = obj`) **binds a name to an object**. It never copies.

```
        x = [1, 2, 3]
        y = x

        x ──┐
            ├──> [1, 2, 3]      ONE list object, TWO labels
        y ──┘
```

**Analogy:** an object is a house; a name is a sticky note with an address on it. `y = x`
copies the *sticky note*, not the house. Both notes now point at the same house — and if
someone repaints it, both notes still lead to the repainted house.

Three tools let you inspect this:

| Tool | Question it answers |
|---|---|
| `id(obj)` | Which object is this? (its address) |
| `a is b` | Are these the *same object*? |
| `a == b` | Do these have the *same value*? |

In [ ]:
a = [1, 2, 3]
b = a                       # binds a SECOND name to the SAME object
c = [1, 2, 3]               # builds a NEW object with an equal value

print("a:", a, "id:", id(a))
print("b:", b, "id:", id(b), " <- identical to a")
print("c:", c, "id:", id(c), " <- different object")

print("\na is b :", a is b, "  (same object)")
print("a is c :", a is c, "  (different objects)")
print("a == c :", a == c, "  (equal values)")

# The consequence: mutating through one name is visible through the other
b.append(4)
print("\nafter b.append(4):")
print("  a:", a, " <- changed too!")
print("  b:", b)
print("  c:", c, " <- untouched, it was never the same object")

### Rebinding vs mutating — the distinction that matters

There are two very different things you can do to `a`:

| | What it does | Visible through aliases? |
|---|---|---|
| `a = [7, 8, 9]` | **Rebinds** the name `a` to a new object | ❌ No |
| `a[0] = 7` | **Mutates** the object `a` points to | ✅ Yes |
| `a.append(4)` | Mutates | ✅ Yes |
| `a += [4]` | Mutates *for lists* (calls `__iadd__`) | ✅ Yes |
| `a = a + [4]` | Builds a new list, rebinds | ❌ No |

That second-to-last row surprises people: for a **list**, `a += [4]` is *not* the same as
`a = a + [4]`.

In [ ]:
# Rebinding: the alias does not follow
a = [1, 2, 3]
b = a
a = [7, 8, 9]
print("rebind  -> a:", a, "| b:", b)

# Mutating: the alias sees it
a = [1, 2, 3]
b = a
a[0] = 99
print("mutate  -> a:", a, "| b:", b)

# += on a LIST mutates in place
a = [1, 2, 3]
b = a
a += [4]
print("a += [4]-> a:", a, "| b:", b, " <- b changed")

# a = a + [4] builds a new list
a = [1, 2, 3]
b = a
a = a + [4]
print("a = a+[4]-> a:", a, "| b:", b, " <- b did not")

# For IMMUTABLE types the question never arises - there is nothing to mutate
s = "hello"
t = s
s += " world"
print("\nstrings -> s:", s, "| t:", t, " <- always safe")

---

## 2. Mutable vs immutable

| | Immutable | Mutable |
|---|---|---|
| **Types** | `int`, `float`, `complex`, `bool`, `str`, `tuple`, `frozenset`, `bytes` | `list`, `dict`, `set`, `bytearray`, most custom classes |
| **Can change after creation?** | No | Yes |
| **Safe to share between names?** | Always | Only if you mean to |
| **Can be a dict key / set element?** | Yes (if contents are too) | No |

Immutability is not a limitation — it is a **guarantee**. If you hold a reference to a
tuple, nobody anywhere in the program can change it under you. That is why immutable
objects are safe to share, safe to cache, safe across threads, and eligible to be hashed.

In [ ]:
# Immutable: every "change" produces a new object
n = 10
print("n =", n, "| id:", id(n))
n += 1
print("n =", n, "| id:", id(n), " <- different object")

s = "abc"
print("\ns =", s, "| id:", id(s))
s += "d"
print("s =", s, "| id:", id(s), " <- different object")

# Mutable: the object itself changes, identity preserved
lst = [1, 2, 3]
print("\nlst =", lst, "| id:", id(lst))
lst.append(4)
print("lst =", lst, "| id:", id(lst), " <- SAME object")

# ⚠️ Immutable containers can still hold mutable objects
t = ([1, 2], "fixed")
print("\nbefore:", t)
t[0].append(3)                  # mutating the list INSIDE the tuple
print("after :", t, " <- the tuple did not change; its contents did")

try:
    t[0] = [9]
except TypeError as exc:
    print("rebinding a slot:", exc)

---

## 3. Copying: shallow vs deep

Once you know that assignment doesn't copy, the obvious question is: how *do* you copy?

There are three levels, and choosing the wrong one is a classic bug.

| | What it does | Nested objects |
|---|---|---|
| `b = a` | No copy at all — a second name | Shared |
| `b = a.copy()` / `a[:]` / `list(a)` / `copy.copy(a)` | **Shallow** — new outer container | **Shared** |
| `b = copy.deepcopy(a)` | **Deep** — recursively copies everything | Independent |

**Analogy:** a shallow copy is photocopying a page of *addresses*. You now have two lists,
but they still point at the same houses. A deep copy builds new houses too.

In [ ]:
import copy

original = [[1, 2], [3, 4], [5, 6]]

alias = original                    # not a copy at all
shallow = original.copy()           # new outer list, SAME inner lists
deep = copy.deepcopy(original)      # everything new

print("outer lists distinct?")
print("  alias  :", alias is original)
print("  shallow:", shallow is original)
print("  deep   :", deep is original)

print("\ninner lists distinct?")
print("  shallow[0] is original[0]:", shallow[0] is original[0], " <- SHARED")
print("  deep[0]    is original[0]:", deep[0] is original[0], " <- independent")

# Mutate a nested element and watch what follows
original[0].append(99)

print("\nafter original[0].append(99):")
print("  original:", original)
print("  alias   :", alias,   " <- same object")
print("  shallow :", shallow, " <- inner list was shared!")
print("  deep    :", deep,    " <- fully independent")

In [ ]:
import copy

# Four equivalent ways to make a SHALLOW copy of a list
data = [1, 2, 3]
print(data.copy(), data[:], list(data), copy.copy(data))

# Dicts and sets have .copy() too
d = {"a": 1, "b": [2, 3]}
d_shallow = d.copy()
d_shallow["b"].append(4)
print("\nshallow dict copy shares the inner list:", d)

d_deep = copy.deepcopy(d)
d_deep["b"].append(99)
print("deep dict copy does not             :", d)

# When is a shallow copy enough? When the contents are IMMUTABLE.
flat = [1, 2, 3, "four", (5, 6)]
flat_copy = flat.copy()
flat_copy.append(7)
print("\nflat original:", flat, " <- unaffected, nothing nested is mutable")

# deepcopy handles cycles correctly - it will not loop forever
cyclic = [1, 2]
cyclic.append(cyclic)
safe = copy.deepcopy(cyclic)
print("\ndeepcopy of a self-referencing list:", safe[2] is safe)

> **Rule of thumb:** if every element is immutable, a shallow copy *is* a full copy.
> The moment you have containers inside containers, decide deliberately.
>
> `deepcopy()` is not free — it walks the entire structure and tracks what it has already
> seen. Don't reach for it by default on large data.

---

## 4. The mutable default argument

This is the single most famous Python gotcha, and it follows directly from everything above.

```python
def add_item(item, basket=[]):     # ⚠️ DO NOT DO THIS
    basket.append(item)
    return basket
```

**Default argument values are evaluated once — when the `def` statement runs**, not on each
call. So every call that doesn't pass `basket` shares *one* list, which accumulates across
calls.

The fix is always the same: use `None` as the sentinel and create the real default inside.

In [ ]:
# The bug
def add_item_buggy(item, basket=[]):
    basket.append(item)
    return basket

print("call 1:", add_item_buggy("apple"))
print("call 2:", add_item_buggy("banana"), " <- apple is still there!")
print("call 3:", add_item_buggy("cherry"))

# Proof that there is exactly one shared list, created at def time
print("\nthe default itself:", add_item_buggy.__defaults__)


# The fix
def add_item(item, basket=None):
    if basket is None:
        basket = []              # a FRESH list on every call
    basket.append(item)
    return basket

print("\ncall 1:", add_item("apple"))
print("call 2:", add_item("banana"), " <- correct")
print("call 3:", add_item("cherry"))

# Passing your own still works
mine = ["existing"]
print("\nexplicit:", add_item("new", mine))

# Immutable defaults are perfectly safe - there is nothing to accumulate
def greet(name, greeting="Hello"):
    return f"{greeting}, {name}!"

print("\n" + greet("Aditya"))

---

## 5. Hashability — the rule behind dict keys and set elements

An object is **hashable** if it has a hash value that never changes during its lifetime.
Dictionaries and sets use that hash to find things in O(1) time.

The rule follows from mutability:

> **Mutable objects are unhashable.** If an object could change after being stored, its
> hash would change too, and the dict would no longer be able to find it.

| Hashable | Unhashable |
|---|---|
| `int`, `float`, `str`, `bytes`, `bool`, `None` | `list` |
| `tuple` — **only if every element is hashable** | `dict` |
| `frozenset` | `set` |
| Custom objects — by default, by identity | Custom objects defining `__eq__` without `__hash__` |

In [ ]:
# Hashable things can be keys and set members
print("hash(42)      :", hash(42))
print("hash('abc')   :", hash("abc"))
print("hash((1, 2))  :", hash((1, 2)))
print("hash(frozenset([1,2])):", hash(frozenset([1, 2])))

# Unhashable things cannot
for obj in ([1, 2], {"a": 1}, {1, 2}):
    try:
        hash(obj)
    except TypeError as exc:
        print(f"\nhash({obj!r}):", exc)

# A tuple is hashable only if its CONTENTS are
print("\nhash((1, 'a', (2, 3))):", hash((1, "a", (2, 3))))
try:
    hash((1, [2, 3]))
except TypeError as exc:
    print("hash((1, [2, 3]))    :", exc)

# The practical payoff: composite keys
sales = {
    ("Mumbai", 2024): 150_000,
    ("Delhi", 2024): 132_000,
    ("Mumbai", 2025): 168_000,
}
print("\nMumbai 2025:", sales[("Mumbai", 2025)])

# ⚠️ And why an int key and a bool key can collide
weird = {1: "one", True: "TRUE", 1.0: "float one"}
print("\n{1: ..., True: ..., 1.0: ...} ->", weird)
print("because hash(1) == hash(True) == hash(1.0) and 1 == True == 1.0")

---

## 6. Nested structures

Real data is rarely flat. A JSON API response, a CSV parsed into records, a config file —
all become dicts containing lists containing dicts.

Two skills matter: **building** them without aliasing accidents, and **reading** them
without a `KeyError` every time a field is missing.

In [ ]:
# Building a nested structure
students = [
    {"name": "Aditya", "marks": {"phy": 88, "chem": 91}, "tags": ["senior"]},
    {"name": "Priya",  "marks": {"phy": 95, "chem": 89}, "tags": ["senior", "topper"]},
    {"name": "Rahul",  "marks": {"phy": 72, "chem": 65}, "tags": []},
]

# Reading: index and key, level by level
print("Priya's physics mark:", students[1]["marks"]["phy"])

# Iterating a nested structure
print()
for s in students:
    total = sum(s["marks"].values())
    avg = total / len(s["marks"])
    tags = ", ".join(s["tags"]) or "-"
    print(f"  {s['name']:<8} avg={avg:5.1f}  tags: {tags}")

# ⚠️ The aliasing trap when building
template = {"marks": {}, "tags": []}
bad = [template.copy() for _ in range(3)]     # shallow! all share ONE marks dict
bad[0]["marks"]["phy"] = 100
print("\nshallow template ->", [b["marks"] for b in bad], " <- all changed")

good = [{"marks": {}, "tags": []} for _ in range(3)]   # fresh objects each time
good[0]["marks"]["phy"] = 100
print("fresh each time  ->", [g["marks"] for g in good])

In [ ]:
data = {"user": {"address": {"city": "Pune"}}}

# Direct access raises as soon as a level is missing
try:
    print(data["user"]["profile"]["city"])
except KeyError as exc:
    print("Direct access:", exc)

# Chained .get() with {} defaults - safe, if a little noisy
city = data.get("user", {}).get("address", {}).get("city")
print("\nchained get   :", city)

missing = data.get("user", {}).get("profile", {}).get("city")
print("missing branch:", missing, " <- None, no exception")


# A reusable helper
def get_nested(mapping, *keys, default=None):
    """Walk a nested mapping, returning `default` if any level is missing."""
    current = mapping
    for key in keys:
        if not isinstance(current, dict) or key not in current:
            return default
        current = current[key]
    return current


print("\nget_nested(data, 'user', 'address', 'city') :", get_nested(data, "user", "address", "city"))
print("get_nested(data, 'user', 'profile', 'city') :", get_nested(data, "user", "profile", "city"))
print("with a default                              :",
      get_nested(data, "user", "profile", "city", default="unknown"))

---

## 7. Unpacking, in full

Unpacking assigns several names from one iterable. You met the basics in **2.3 Tuple**;
here is the complete picture, including the dict forms.

### Sequence unpacking

```
first, *middle, last = [1, 2, 3, 4, 5]
```

### In function calls — `*` and `**`

| Syntax | In a **call** | In a **definition** |
|---|---|---|
| `*args` | Spread a sequence into positional arguments | Collect extra positionals into a tuple |
| `**kwargs` | Spread a mapping into keyword arguments | Collect extra keywords into a dict |

(Function definitions are covered in **04 Functions** — this is the calling side.)

In [ ]:
# ---- Sequence unpacking ----
a, b, c = [1, 2, 3]
print("basic   :", a, b, c)

first, *rest = "python"
print("starred :", first, rest)

(name, age), city = ("Aditya", 25), "Pune"
print("nested  :", name, age, city)

# Swap without a temp
x, y = 1, 2
x, y = y, x
print("swapped :", x, y)


# ---- Unpacking into a function CALL ----
def describe(name, age, city):
    return f"{name}, {age}, from {city}"

person_tuple = ("Priya", 30, "Delhi")
print("\n*tuple  :", describe(*person_tuple))

person_dict = {"name": "Rahul", "age": 28, "city": "Mumbai"}
print("**dict  :", describe(**person_dict))


# ---- Unpacking to BUILD containers ----
a1, a2 = [1, 2], [3, 4]
print("\nmerged list :", [*a1, *a2])
print("merged tuple:", (*a1, *a2))
print("merged set  :", {*a1, *a2})

d1, d2 = {"a": 1}, {"b": 2}
print("merged dict :", {**d1, **d2})
print("with override:", {**d1, **d2, "a": 99})


# ---- Unpacking in loops ----
pairs = [("phy", 88), ("chem", 91)]
for subject, mark in pairs:
    print(f"\n  {subject}: {mark}", end="")

print("\n")
for i, (subject, mark) in enumerate(pairs, start=1):
    print(f"  {i}. {subject} = {mark}")

# zip() + unpacking: transpose
rows = [(1, 2, 3), (4, 5, 6)]
print("\ntransposed:", list(zip(*rows)))

---

## Common Mistakes & Pitfalls

1. **Thinking `b = a` copies.** It binds a second name to the same object. Only `copy()` or `deepcopy()` copy.
2. **Using `.copy()` on nested data.** It is shallow — inner objects stay shared. Use `copy.deepcopy()`.
3. **A mutable default argument** (`def f(x, items=[])`). Use `None` as the sentinel.
4. **`[[0] * 3] * 3` for a grid.** The outer `*` repeats a *reference*. Use a comprehension.
5. **`dict.fromkeys(keys, [])`.** Every key gets the same list. Use a dict comprehension.
6. **Assuming `a += b` and `a = a + b` are equivalent.** For lists the first mutates in place (visible to aliases), the second rebinds.
7. **Trying to use a list as a dict key.** Convert to a tuple.
8. **Using `is` to compare values.** Use `==`; reserve `is` for `None`, `True`, `False`.

## Best Practices

- Default to **immutable** types unless you need to mutate — they are safe to share.
- Build fresh objects in a comprehension rather than copying a template.
- Use `None` as the default for any parameter whose real default is mutable.
- Reach for `deepcopy()` deliberately, not by reflex — it is expensive on large structures.
- Use tuples as composite dict keys instead of nesting dicts two levels deep.
- Use unpacking (`x, y = point`) rather than indexing — it documents the shape.
- When a function must not modify its argument, copy it at the top of the function and say so in the docstring.

## Practice Exercises

Try these before moving on.

1. Predict the output, then verify: `a = [1,[2,3]]; b = a.copy(); b[1].append(4); print(a)`.
2. Write `duplicate(grid)` that copies a 2-D list so the copy is fully independent. Prove it with a mutation.
3. Fix this: `def append_to(x, target=[]): target.append(x); return target`.
4. Build a dict mapping each of `'a','b','c'` to its own empty list — first the broken way with `fromkeys`, then correctly.
5. Explain why `x = (1, 2); x += (3,)` works even though tuples are immutable.
6. Given nested JSON-like data, write a function returning all values for a given key at any depth.
7. Using only unpacking, swap the first and last elements of a list in one statement.
8. Show two objects where `a == b` is `True` but `a is b` is `False`, and two where both are `True`.